# Model the Air Quality Index with a gaussian process.

In [18]:
import os
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from MuyGPyS._test.sampler import print_results
from MuyGPyS.neighbors import NN_Wrapper
from MuyGPyS.gp import MuyGPS
from MuyGPyS.gp.deformation import Isotropy, l2, Anisotropy
from MuyGPyS.gp.hyperparameter import AnalyticScale, Parameter, VectorParameter
from MuyGPyS.gp.kernels import Matern
from MuyGPyS.gp.noise import HomoscedasticNoise
from MuyGPyS.optimize import Bayes_optimize, L_BFGS_B_optimize
from MuyGPyS.optimize.batch import sample_batch
from MuyGPyS.optimize.loss import lool_fn, mse_fn, pseudo_huber_fn, looph_fn
import pandas as pd
import time

# Import and setup data

In [19]:
BASE_DIR = os.path.abspath("..") + "\\"
aqi_data = np.genfromtxt(BASE_DIR + "data\\biggest_day_aqi_coords_scaled_noAH.csv", delimiter=',', skip_header=1)
#print(aqi_data[:10])

In [20]:
#randomly split the data into features and responces, for later splitting into folds
aqi_x, aqi_y = aqi_data[:, :-1], aqi_data[:, -1]

In [21]:
kfold = KFold(n_splits=10, shuffle=True, random_state=24)

# Nearest Neighbor and Batches

# setting and optimizing hyperparamters

In [22]:
#create our unoptimized muyGP object. set the ranges we want the hyperparameters to be optimized between.
aqi_muygps = MuyGPS(
    kernel=Matern(
        smoothness=Parameter("log_sample", (0.5, 2.5)),
        deformation=Isotropy(
            l2,
            length_scale=Parameter("log_sample", (0.01, 0.20))
        ),
    ),
    noise=HomoscedasticNoise("sample", (0.003, 0.25)),
    scale=AnalyticScale(),
)

In [23]:
def optimize_gp():
    #use the indices of the training points, and of the training points neighbors, together with the points' coordinates and values
    #to calculate the crosswise distances between points and the pairwise distances between the neighbors
    #and also the actual value of each point, and each point's neighbor.
    (
        batch_crosswise_dists,
        batch_pairwise_dists,
        batch_ys,
        batch_nn_ys,
    ) = aqi_muygps.make_train_tensors(
        batch_indices,
        batch_nn_indices,
        aqi_x_train,
        aqi_y_train,
    )
                  
    optimize_start = time.time()
    #optimize the parameters for 15 random initilization runs, and then 25 optimizing runs.
    # aqi_muygps_optimized = Bayes_optimize(
    #     aqi_muygps,
    #     batch_ys,
    #     batch_nn_ys,
    #     batch_crosswise_dists,
    #     batch_pairwise_dists,
    #     loss_fn=lool_fn,
    #     verbose=True,
    #     random_state=24,
    #     init_points=20,
    #     n_iter=30,
    # )

    aqi_muygps_optimized = L_BFGS_B_optimize(
        aqi_muygps,
        batch_ys,
        batch_nn_ys,
        batch_crosswise_dists,
        batch_pairwise_dists,
        loss_fn=lool_fn,
        verbose=True,
    )
    
    #also optimize the scale
    aqi_muygps_optimized = aqi_muygps_optimized.optimize_scale(
        batch_pairwise_dists,
        batch_nn_ys
    )
    optimize_end = time.time()
    optimize_time = optimize_end - optimize_start
    print(f"Gaussian Process optimization took {optimize_time} seconds")
    return aqi_muygps_optimized, optimize_time

# Inference

In [24]:
def predict_gp():
    #create tesors to store the crosswise distances and pairwise distances between points and the point's neighbors.
    #as well as the actual values for each points neighbors.
    (
        test_crosswise_dists,
        test_pairwise_dists,
        test_nn_ys,
    ) = aqi_muygps.make_predict_tensors(
        test_indices,
        test_nn_indices,
        aqi_x_test,
        aqi_x_train,
        aqi_y_train,
    )

    #calculate kernels to store the crosswise covariance and pairwise covariance. these are used to know how much a points neighbors influence it.
    kcross = aqi_muygps_optimized.kernel(test_crosswise_dists)
    kin = aqi_muygps_optimized.kernel(test_pairwise_dists)

    #Take the crosswise covariance and pairwise covariance to know how the neighbors effect each point,
    #and then use the values of the points to calculate the actual value based on the neighbors influence.
    predictions = aqi_muygps_optimized.posterior_mean(kin, kcross, test_nn_ys)
    #because mean was removed from the datapoints before, need to add it back in.
    predictions = predictions + aqi_y_mean
    #calculate the variances, how much the guess is spread, meaning uncertainty.
    variances = aqi_muygps_optimized.posterior_variance(kin, kcross)

    return predictions, variances

# Metrics

In [25]:
def metrics_gp():
    rmse = np.sqrt(np.mean((aqi_y_test - predictions) ** 2))
    #get the range that values need to be in in order to fall within 95% certainty.
    confidence_intervals = np.sqrt(variances) * 1.96
    #coverage is the proportion of guesses that differ from the true response by no more than the confidence interval size.
    coverage = np.count_nonzero(np.abs(aqi_y_test - predictions) < confidence_intervals) / test_count

    rmse_scores.append(rmse)
    mean_variances.append(np.mean(variances))
    mean_conf_intervals.append(np.mean(confidence_intervals)) 
    optimization_times.append(optimize_time)

    #print the results of the model
    print_results(
        aqi_y_test, ("optimized", aqi_muygps_optimized, predictions, variances, confidence_intervals, coverage)
    )

In [26]:
#setup metrics storage, to keep track of performance
optimization_times = []
rmse_scores = []
mean_variances = []
mean_conf_intervals = []

In [27]:
#cycle through all folds
for fold, (train_idx, test_idx) in enumerate(kfold.split(aqi_x)):
    #seperate into training testing feature and value sets.
    aqi_x_train, aqi_x_test = aqi_x[train_idx], aqi_x[test_idx]
    aqi_y_train, aqi_y_test = aqi_y[train_idx], aqi_y[test_idx]
    #use the mean of the training points to provide basic de-meaning of the training points.
    aqi_y_mean = aqi_y_train.mean()
    aqi_y_train = aqi_y_train - aqi_y_mean

    #setup nearest neigbor
    nn_count = 30
    nbrs_lookup = NN_Wrapper(aqi_x_train, nn_count, nn_method="exact", algorithm="ball_tree")
    #for the sake of making use of batches, will use a batch value of 1500. this will include all of the data points.
    batch_count = 1500
    #get the indices of the training points and the indices of each points neighbors.
    batch_indices, batch_nn_indices = sample_batch(
        nbrs_lookup, batch_count, len(aqi_x_train)
    )

    #get test equivalents of above
    test_count = aqi_x_test.shape[0]
    #get the indices of all the test points
    test_indices = np.arange(test_count)
    #get the indices of the neighbors for each of those test points as well
    test_nn_indices, _ = nbrs_lookup.get_nns(aqi_x_test)

    #first optimize the model
    aqi_muygps_optimized, optimize_time = optimize_gp()

    #now predict values
    predictions, variances = predict_gp()

    #gather metrics
    metrics_gp()

#print out the results
print(f"Avg RMSE: {np.mean(rmse_scores):.4f}")
print(f"Avg Variances: {np.mean(mean_variances):.4f}")
print(f"Avg Confidence Intervals: {np.mean(mean_conf_intervals):.4f}")
print(f"Avg Optimization Time: {np.mean(optimization_times):.4f}")

parameters to be optimized: ['length_scale', 'smoothness', 'noise']
bounds: [[0.01  0.2  ]
 [0.5   2.5  ]
 [0.003 0.25 ]]
initial x0: [0.03421131 1.24497721 0.10549053]
optimizer results: 
  message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  success: True
   status: 0
      fun: 6181.618677209466
        x: [ 1.514e-02  5.000e-01  2.500e-01]
      nit: 8
      jac: [-5.457e-04  2.551e+02 -2.051e+02]
     nfev: 40
     njev: 10
 hess_inv: <3x3 LbfgsInvHessProduct with dtype=float64>
Gaussian Process optimization took 11.254860639572144 seconds
parameters to be optimized: ['length_scale', 'smoothness', 'noise']
bounds: [[0.01  0.2  ]
 [0.5   2.5  ]
 [0.003 0.25 ]]
initial x0: [0.03421131 1.24497721 0.10549053]
optimizer results: 
  message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  success: True
   status: 0
      fun: 6240.4464923725545
        x: [ 1.577e-02  5.000e-01  2.500e-01]
      nit: 8
      jac: [-9.095e-05  3.253e+02 -3.489e+02]
     nfev: 40
     nj

In [28]:
predictions_df = pd.DataFrame(predictions)
predictions_df.describe()



,0
count,108.000000
mean,46.451160
std,9.373837
min,13.492805
25%,41.760011
50%,47.262026
75%,51.472915
max,68.258192


In [29]:
confidence_intervals = np.sqrt(variances) * 1.96
#coverage is the proportion of guesses that differ from the true response by no more than the confidence interval size.
coverage = np.count_nonzero(np.abs(aqi_y_test - predictions) < confidence_intervals) / test_count
print_results(
    aqi_y_test, ("optimized", aqi_muygps_optimized, predictions, variances, confidence_intervals, coverage)
)

name,smoothness,length scale,noise variance,variance scale,rmse,mean variance,mean confidence interval,coverage
optimized,0.500000,0.015839,0.250000,223.520148,14.021861,138.918937,22.725859,0.898148
